# VoD 통합 비교 노트북 — **구조적 위치: B (오프라인 실험/검증)**

이 파일은 **최종 운용 파이프라인이 아니라**, 데이터셋(VoD) 위에서 **레이더·카메라·(보조) LiDAR**를 **나란히 비교**하는 **오프라인 실험**용입니다.

---

## A. 최종 시스템 (발표·웹에서 말하는 축)

**가정: FMCW 레이더 단독 입력**

1. 전처리  
2. 군집화(DBSCAN 등)  
3. 후보 선택  
4. 추적  
5. 위험도 계산  

운영 UI·로직에서는 **LiDAR를 입력으로 쓰지 않습니다.**  
후보의 **신뢰도**는 `strong_validated` / `radar_only` 같은 *LiDAR 검증 기반 라벨* 대신, **레이더 내부 특징만**으로 등급을 나누는 편이 말이 깔끔합니다.

**권장 등급 예시**

- `high_confidence` / `medium_confidence` / `low_confidence`

**등급에 쓸 수 있는 레이더 특징 (LiDAR 불필요)**

- 군집 점 수  
- RCS 안정성  
- 속도 일관성  
- 프레임 간 추적 지속성  
- heading 변화 안정성  

---

## B. 오프라인 실험·검증 (이 노트북이 속하는 쪽)

- 동일 후보(또는 동일 프레임)에 대해 **LiDAR·GT(`label_2`)·카메라**로 **교차 검증**  
- **레이더-only 방식의 한계**·임계값 튜닝 **참고**  
- 보고서·노트북에는 LiDAR 포함 가능 — 단, **「실제 시스템 입력 아님 / 오프라인 비교」**를 명시  

---

## 발표에서 쓰기 안전한 문장 (복사용)

- *「최종 운용 가정은 FMCW 레이더 단독이며, LiDAR는 실제 시스템 입력이 아니라 데이터셋 기반 오프라인 검증용 보조 센서로만 활용했습니다.»*  
- *「LiDAR 검증은 레이더 후보의 공간적 타당성을 확인하기 위한 실험적 비교 단계이며, 실제 배치 파이프라인에서는 제외됩니다.»*


---

## 한 줄 결론

**FMCW 레이더만 실제로 쓸 계획**이라면, LiDAR 검증은 **최종 시스템 핵심 로직처럼 말하거나 운영 UI의 메인 구분으로 두지 말고**, **오프라인 실험·성능 비교·후보 타당성 확인용 보조 검증**으로만 쓰면 됩니다. 운영 신뢰도는 `high_confidence` / `medium_confidence` / `low_confidence` 등 **레이더 특징 기반**으로 표현합니다.


---

## 아래 셀에서 하는 일 (단계 요약)

| 단계 | 내용 | 비고 |
|------|------|------|
| 1 | 설정·프레임·**원본 카메라** | 발표용 참고 영상 |
| 2 | (선택) AI 서버 **융합 API** | 데모/비교용; LiDAR 교차검증이 섞일 수 있음 → **B 구간** |
| 3 | **BEV CNN** — LiDAR BEV vs **레이더만 BEV** | 동일 `label_2` 히트맵, **오프라인** 성능 비교 |
| 4 | **label_2** GT 바닥면 폴리곤 | 지도 학습·평가 기준 |
| 5 | **LiDAR DBSCAN** BEV 박스 | 규칙 기반 **참고** |
| 6 | **2×4** 통합 그림 | 한 프레임 **정성 비교** |

**필요**: `vod-received/view_of_delft_PUBLIC`(또는 `VOD_ROOT`), `torch`, `numpy`, `matplotlib`, `sklearn`, `requests`, `PIL` · (선택) `ai-inference`  

모듈: `bev_lidar_detector_train.py`, `vod_compare_utils.py`


## 0. 공통 import


In [ ]:
from __future__ import annotations

import io
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Polygon
import numpy as np
import torch
from PIL import Image
from sklearn.cluster import DBSCAN

NOTEBOOK_DIR = Path.cwd().resolve()
sys.path.insert(0, str(NOTEBOOK_DIR))

import bev_lidar_detector_train as bev
import vod_compare_utils as vcu

# Windows 한글 (선택)
try:
    from matplotlib import font_manager, rc
    _fp = Path("C:/Windows/Fonts/malgun.ttf")
    if _fp.is_file():
        _fn = font_manager.FontProperties(fname=str(_fp)).get_name()
        rc("font", family=_fn)
        rc("axes", unicode_minus=False)
except Exception:
    pass

print("ready:", NOTEBOOK_DIR)


## 1. 설정: 데이터 루트 · 프레임 ID · AI 서버 URL


In [ ]:
DEFAULT_ROOT = NOTEBOOK_DIR / "vod-received" / "view_of_delft_PUBLIC"
DATASET_ROOT = Path(os.environ.get("VOD_ROOT", str(DEFAULT_ROOT))).resolve()
AI_INFERENCE_URL = os.environ.get("AI_INFERENCE_URL", "http://127.0.0.1:8001")

# 비교할 프레임 (동기 stem). 예: "01201"
FRAME_ID = os.environ.get("VOD_COMPARE_FRAME", "01201")

lidar_dir = DATASET_ROOT / "lidar" / "training" / "velodyne"
radar_dir = DATASET_ROOT / "radar" / "training" / "velodyne"
image_dir = DATASET_ROOT / "lidar" / "training" / "image_2"
label_dir = DATASET_ROOT / "lidar" / "training" / "label_2"
calib_dir = DATASET_ROOT / "lidar" / "training" / "calib"
if not calib_dir.is_dir():
    calib_dir = DATASET_ROOT / "calib"

paths = {
    "lidar": lidar_dir / f"{FRAME_ID}.bin",
    "radar": radar_dir / f"{FRAME_ID}.bin",
    "image": image_dir / f"{FRAME_ID}.jpg",
    "label": label_dir / f"{FRAME_ID}.txt",
    "calib": calib_dir / f"{FRAME_ID}.txt",
}

for k, p in paths.items():
    print(k, "OK" if p.is_file() else "MISSING", p)

if not paths["lidar"].is_file():
    raise FileNotFoundError("LiDAR 없음 — FRAME_ID 또는 VOD_ROOT 확인")


## 2. 원본 카메라 이미지 보기


In [ ]:
if paths["image"].is_file():
    img_raw = Image.open(paths["image"]).convert("RGB")
    plt.figure(figsize=(10, 5))
    plt.imshow(img_raw)
    plt.title(f"원본 카메라 — frame {FRAME_ID}")
    plt.axis("off")
    plt.tight_layout()
    plt.show()
else:
    img_raw = None
    print("이미지 파일 없음")


## 3. AI 서버: 레이더 DBSCAN + YOLO + (데이터셋) LiDAR 융합 (`/infer/vod/radar-fusion`)

**위치: B (오프라인·데모 비교)** — 최종 FMCW-only 배치 파이프라인에 LiDAR 입력을 넣는다고 가정하지 않습니다.

서버 응답에 LiDAR 교차검증이 포함될 수 있으면, 그것은 **후보의 공간적 타당성을 보는 실험적 단계**로 해석하면 됩니다. 서버가 꺼져 있으면 `FUSION = None`이고 이후는 로컬 레이더·LiDAR 파일로만 B 실험을 진행합니다.


In [ ]:
FUSION: dict | None = None
FUSION_ERROR: str | None = None

if paths["radar"].is_file():
    try:
        prev_id = str(max(0, int(FRAME_ID) - 1)).zfill(len(FRAME_ID))
        prev_radar = radar_dir / f"{prev_id}.bin"
        if not prev_radar.is_file():
            prev_radar = None
        FUSION = vcu.post_vod_radar_fusion(
            AI_INFERENCE_URL,
            paths["radar"],
            paths["image"] if paths["image"].is_file() else None,
            paths["lidar"],
            radar_prev_path=prev_radar,
        )
        print("fusion ok:", FUSION.get("ok"), "inferMs:", FUSION.get("inferMs"))
        print("radarDetections:", len(FUSION.get("radarDetections") or []))
        print("yoloDetections:", len(FUSION.get("yoloDetections") or []))
    except Exception as e:
        FUSION_ERROR = str(e)
        print("AI 융합 스킵:", FUSION_ERROR)
else:
    print("레이더 파일 없음 — 융합 생략")


### 3-1. YOLO 결과 이미지 (서버가 그린 박스)


In [ ]:
if FUSION and FUSION.get("annotatedImageBase64"):
    yolo_vis = vcu.b64_to_pil_image(FUSION["annotatedImageBase64"])
    plt.figure(figsize=(10, 5))
    plt.imshow(yolo_vis)
    plt.title("YOLO 검출 (ai-inference)")
    plt.axis("off")
    plt.tight_layout()
    plt.show()
elif paths["image"].is_file():
    try:
        solo = vcu.post_yolo_image_only(AI_INFERENCE_URL, paths["image"])
        if solo.get("annotatedImageBase64"):
            plt.figure(figsize=(10, 5))
            plt.imshow(vcu.b64_to_pil_image(solo["annotatedImageBase64"]))
            plt.title("YOLO only (/infer/image)")
            plt.axis("off")
            plt.show()
    except Exception as e:
        print("YOLO 단독 호출 실패:", e)
else:
    print("표시할 YOLO 이미지 없음")


## 4. LiDAR / 레이더 / 캘리브 / 라벨 로드 (로컬) — **오프라인 검증용 데이터**

VoD는 **레이더·LiDAR·카메라·라벨**이 같이 있어, **같은 프레임**에서 A축(레이더 후보)과 **GT/LiDAR 규칙**을 나란히 놓을 수 있습니다.  
**최종 시스템**에서는 이 LiDAR 스트림이 없다고 보고, 여기서 나온 비교는 **B: 보조 검증·한계 파악·튜닝 참고**로만 쓰면 됩니다.


In [ ]:
lidar_pts = bev.parse_lidar_bin(paths["lidar"])
calib = bev.parse_calib_txt(paths["calib"])
label_rows = bev.parse_kitti_label(paths["label"]) if paths["label"].is_file() else []
radar_pts = np.fromfile(paths["radar"], dtype=np.float32).reshape(-1, 7) if paths["radar"].is_file() else np.zeros((0, 7))

print("LiDAR points:", lidar_pts.shape[0], "| Radar points:", radar_pts.shape[0], "| label rows:", len(label_rows))
print("calib:", calib is not None)

bev_cfg = bev.BevGridConfig()
gt_footprints = vcu.label_rows_to_velo_footprints(label_rows, calib) if calib else []


## 5. BEV CNN 학습 (LiDAR 입력 vs 레이더만 입력) — **오프라인 비교 실험**

동일 **BCE + label_2 히트맵**으로 두 모델을 각각 학습해, **「같은 지도학습 타깃을 맞출 때 입력이 LiDAR BEV냐 레이더 BEV냐」**만 바꿔 봅니다.  
**최종 운용**에서는 레이더 쪽 모델·후체만 의미가 있고, LiDAR BEV 모델은 **상한·참고(upper bound / ablation)**에 가깝습니다.

- **LiDAR BEV**: 점밀도·높이 등 LiDAR 기반 BEV → CNN  
- **레이더만 BEV**: `build_bev_tensor_radar` (동일 그리드, 레이더 점만)

**환경 변수**  
`BEV_QUICK_MAX_TRAIN`(200), `BEV_QUICK_MAX_VAL`(80), `BEV_QUICK_EPOCHS`(8), `BEV_QUICK_BATCH`(4), `BEV_LR`(8e-4), `BEV_WEIGHT_DECAY`(0.02)

학습: AdamW + cosine + grad clip, 에폭마다 train/val BCE 출력.


In [ ]:
from torch.utils.data import DataLoader

frames_all = bev.list_vod_sync_frames(DATASET_ROOT)
splits = bev.split_indices(len(frames_all), 0.7, 0.15)
max_train = int(os.environ.get("BEV_QUICK_MAX_TRAIN", "200"))
max_val = int(os.environ.get("BEV_QUICK_MAX_VAL", "80"))
epochs = int(os.environ.get("BEV_QUICK_EPOCHS", "8"))
batch = int(os.environ.get("BEV_QUICK_BATCH", "4"))
lr = float(os.environ.get("BEV_LR", "8e-4"))
wd = float(os.environ.get("BEV_WEIGHT_DECAY", "0.02"))

if len(frames_all) == 0:
    raise RuntimeError("동기 프레임 없음")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pos_w = torch.full((3, 1, 1), 80.0, device=device)


def _take(inds, cap):
    return [frames_all[int(i)] for i in inds[:cap]]


train_f = _take(splits["train"], max_train)
val_f = _take(splits["valid"], max_val)

# ----- LiDAR 입력 BEV -----
train_ld = bev.BevLidarDataset(train_f, bev_cfg, 3)
val_ld = bev.BevLidarDataset(val_f, bev_cfg, 3)
if len(train_ld) == 0:
    raise RuntimeError(
        "LiDAR 학습 샘플 0 — calib+label_2 있는 프레임이 train split에 있는지 확인"
    )
loader_ld_tr = DataLoader(train_ld, batch_size=batch, shuffle=True, num_workers=0)
loader_ld_va = (
    DataLoader(val_ld, batch_size=batch, shuffle=False, num_workers=0)
    if len(val_ld)
    else None
)

model_lidar = bev.TinyBevDetector(in_ch=2, num_classes=3, base=48).to(device)
opt_ld = torch.optim.AdamW(model_lidar.parameters(), lr=lr, weight_decay=wd)
sched_ld = torch.optim.lr_scheduler.CosineAnnealingLR(opt_ld, T_max=max(epochs, 1))
hist_lidar = []
for ep in range(epochs):
    tr = bev.train_one_epoch(
        model_lidar, loader_ld_tr, opt_ld, device, pos_weight=pos_w, grad_clip_norm=1.0
    )
    va = (
        bev.eval_bce_epoch(model_lidar, loader_ld_va, device, pos_weight=pos_w)
        if loader_ld_va is not None
        else float("nan")
    )
    sched_ld.step()
    hist_lidar.append((ep + 1, tr, va))
    print(f"[LiDAR BEV] ep {ep+1}/{epochs}  train_bce={tr:.5f}  val_bce={va:.5f}")

model_lidar.eval()

# ----- 레이더만 입력 (동일 타깃) -----
train_rd = bev.BevRadarDataset(train_f, bev_cfg, 3)
val_rd = bev.BevRadarDataset(val_f, bev_cfg, 3)
model_radar = None
hist_radar = []
if len(train_rd) == 0:
    print(
        "경고: 레이더 BEV 학습 샘플 없음 — 동기 radar 파일·calib+label_2 확인. RADAR-CNN 스킵."
    )
else:
    loader_rd_tr = DataLoader(train_rd, batch_size=batch, shuffle=True, num_workers=0)
    loader_rd_va = (
        DataLoader(val_rd, batch_size=batch, shuffle=False, num_workers=0)
        if len(val_rd)
        else None
    )
    model_radar = bev.TinyBevDetector(in_ch=2, num_classes=3, base=48).to(device)
    opt_rd = torch.optim.AdamW(model_radar.parameters(), lr=lr, weight_decay=wd)
    sched_rd = torch.optim.lr_scheduler.CosineAnnealingLR(opt_rd, T_max=max(epochs, 1))
    for ep in range(epochs):
        tr = bev.train_one_epoch(
            model_radar, loader_rd_tr, opt_rd, device, pos_weight=pos_w, grad_clip_norm=1.0
        )
        va = (
            bev.eval_bce_epoch(model_radar, loader_rd_va, device, pos_weight=pos_w)
            if loader_rd_va is not None
            else float("nan")
        )
        sched_rd.step()
        hist_radar.append((ep + 1, tr, va))
        print(f"[RADAR BEV] ep {ep+1}/{epochs}  train_bce={tr:.5f}  val_bce={va:.5f}")
    model_radar.eval()

_last_ld = hist_lidar[-1][2] if hist_lidar else float("nan")
_last_rd = hist_radar[-1][2] if hist_radar else float("nan")
print(
    "BEV CNN ready | device:",
    device,
    "| last val BCE  LiDAR:",
    _last_ld,
    "  RADAR:",
    _last_rd,
)


## 6. 현재 프레임 BEV 추론 (LiDAR-CNN · RADAR-CNN) + 레이더 중심 + LiDAR DBSCAN

**B (오프라인)** 한 프레임에서 학습된 두 BEV 모델의 피크와 규칙 기반 LiDAR DBSCAN을 함께 봅니다. 운용 파이프라인의 필수 단계는 아닙니다.


In [ ]:
# BEV 추론 — LiDAR 학습 모델
x_np = bev.build_bev_tensor(lidar_pts, bev_cfg)
with torch.no_grad():
    logits_ld = model_lidar(torch.from_numpy(x_np).unsqueeze(0).to(device))
    prob_ld = torch.sigmoid(logits_ld).squeeze(0).cpu().numpy()

class_names = ["Vehicle", "Pedestrian", "Cyclist"]
peaks_lidar_by_c = []
for c in range(3):
    peaks_lidar_by_c.append(
        bev.heatmap_peaks_xy(torch.from_numpy(prob_ld[c]).float(), bev_cfg, thresh=0.28)
    )

# 레이더만 학습 모델 (가중치가 있을 때)
peaks_radar_by_c = [[] for _ in range(3)]
if model_radar is not None and radar_pts.size > 0:
    x_rd = bev.build_bev_tensor_radar(radar_pts, bev_cfg)
    with torch.no_grad():
        logits_rd = model_radar(torch.from_numpy(x_rd).unsqueeze(0).to(device))
        prob_rd = torch.sigmoid(logits_rd).squeeze(0).cpu().numpy()
    for c in range(3):
        peaks_radar_by_c[c] = bev.heatmap_peaks_xy(
            torch.from_numpy(prob_rd[c]).float(), bev_cfg, thresh=0.28
        )

# 레이더 XY (융합 결과 우선, 없으면 원시 점 평균 군집 대신 간단히 점 투영)
radar_xy = []
if FUSION and FUSION.get("radarDetections"):
    for d in FUSION["radarDetections"]:
        c = d.get("centroidM") or d.get("centroid_m")
        if c and len(c) >= 2:
            radar_xy.append((float(c[0]), float(c[1])))
elif radar_pts.size > 0:
    xy = radar_pts[:, :2]
    m = (
        (xy[:, 0] >= bev_cfg.x_min)
        & (xy[:, 0] <= bev_cfg.x_max)
        & (np.abs(xy[:, 1]) <= max(abs(bev_cfg.y_min), abs(bev_cfg.y_max)))
    )
    sub = xy[m]
    if len(sub) > 20:
        db = DBSCAN(eps=1.2, min_samples=4).fit(sub)
        for lab in set(db.labels_):
            if lab < 0:
                continue
            radar_xy.append(tuple(sub[db.labels_ == lab].mean(axis=0).tolist()))

# LiDAR DBSCAN → BEV 축정렬 박스
lidar_boxes = []
xyz = lidar_pts[:, :3]
m = (
    (xyz[:, 0] >= bev_cfg.x_min)
    & (xyz[:, 0] <= bev_cfg.x_max)
    & (xyz[:, 1] >= bev_cfg.y_min)
    & (xyz[:, 1] <= bev_cfg.y_max)
)
sub = xyz[m]
if len(sub) > 30:
    db2 = DBSCAN(eps=0.9, min_samples=10).fit(sub[:, :2])
    for lab in set(db2.labels_):
        if lab < 0:
            continue
        blk = sub[db2.labels_ == lab]
        xm, ym = blk[:, 0].min(), blk[:, 1].min()
        xM, yM = blk[:, 0].max(), blk[:, 1].max()
        lidar_boxes.append(((xm, ym), (xM, yM)))

print("radar centers (plot):", len(radar_xy), "| lidar cluster boxes:", len(lidar_boxes))
for c, px in enumerate(peaks_lidar_by_c):
    print(f"  LiDAR-CNN peaks {class_names[c]}:", len(px))
for c, px in enumerate(peaks_radar_by_c):
    print(f"  RADAR-CNN peaks {class_names[c]}:", len(px))


## 7. 나란히 최종 비교 (2×4) — **오프라인 정성 비교 그림**

이 그림은 **발표용 “한 장 요약”**에 쓰기 좋지만, **최종 시스템 아키텍처 다이어그램으로 오해하지 않도록** “B 구간 실험”임을 말로 덧붙이는 것을 권장합니다.

- **코드/표가 예전처럼 보이면** 탭을 닫았다가 이 파일을 다시 여세요.  
- **그림이 예전 같으면** 커널 **Restart** 후 위에서부터 순서대로 실행하세요. §5에서 `model_lidar`, `model_radar`가 정의됩니다.  
- **2행×4열**: 아래 줄 오른쪽 두 칸 = **LiDAR 입력 CNN(④)** vs **레이더만 CNN(⑤)**. ⑤에 "RADAR-CNN 없음"이면 `BevRadarDataset` 샘플 0개(동기 `radar`·`label_2`)를 확인하세요.


In [ ]:
def plot_bev_background(ax, pts_xyz, cfg, max_pts=12000):
    m = (
        (pts_xyz[:, 0] >= cfg.x_min)
        & (pts_xyz[:, 0] <= cfg.x_max)
        & (pts_xyz[:, 1] >= cfg.y_min)
        & (pts_xyz[:, 1] <= cfg.y_max)
    )
    p = pts_xyz[m, :2]
    if p.shape[0] > max_pts:
        rng = np.random.default_rng(0)
        p = p[rng.choice(p.shape[0], max_pts, replace=False)]
    ax.scatter(p[:, 1], p[:, 0], s=0.15, c="0.6", alpha=0.35, rasterized=True)
    ax.set_xlim(cfg.y_max, cfg.y_min)
    ax.set_ylim(cfg.x_min, cfg.x_max)
    ax.set_aspect("equal")
    ax.set_xlabel("Y (velo, m)")
    ax.set_ylabel("X (velo, m)")


def _bev_panel_tag(ax, tag: str) -> None:
    ax.text(
        0.02,
        0.98,
        tag,
        transform=ax.transAxes,
        fontsize=10,
        fontweight="bold",
        color="white",
        va="top",
        ha="left",
        bbox=dict(boxstyle="round,pad=0.2", facecolor="black", alpha=0.5),
        zorder=20,
    )


fig = plt.figure(figsize=(19, 9.2))
gs = fig.add_gridspec(2, 4, hspace=0.32, wspace=0.24)

# ----- row 0 -----
ax = fig.add_subplot(gs[0, 0])
if img_raw is not None:
    ax.imshow(img_raw)
    ax.set_title("① 원본 카메라")
else:
    ax.text(0.5, 0.5, "no image", ha="center")
ax.axis("off")

ax = fig.add_subplot(gs[0, 1])
if FUSION and FUSION.get("annotatedImageBase64"):
    ax.imshow(vcu.b64_to_pil_image(FUSION["annotatedImageBase64"]))
    ax.set_title("② YOLO (융합 API)")
elif img_raw is not None:
    ax.imshow(img_raw)
    ax.set_title("② YOLO (미연결 — 원본)")
else:
    ax.text(0.5, 0.5, "YOLO N/A", ha="center")
ax.axis("off")

ax = fig.add_subplot(gs[0, 2:])
ax.axis("off")
val_l_txt = f"{hist_lidar[-1][2]:.5f}" if hist_lidar else "n/a"
val_r_txt = f"{hist_radar[-1][2]:.5f}" if hist_radar else "n/a"
lines = [
    "[요약] 이 그림은 2행×4열 그리드입니다.",
    f"frame: {FRAME_ID}",
    f"LiDAR pts: {lidar_pts.shape[0]} | Radar pts: {radar_pts.shape[0]}",
    f"GT boxes (label_2): {len(gt_footprints)}",
    f"마지막 epoch val BCE — LiDAR-CNN: {val_l_txt} | RADAR-CNN: {val_r_txt}",
    f"LiDAR-CNN peaks V/P/C: {len(peaks_lidar_by_c[0])}/{len(peaks_lidar_by_c[1])}/{len(peaks_lidar_by_c[2])}",
    f"RADAR-CNN peaks V/P/C: {len(peaks_radar_by_c[0])}/{len(peaks_radar_by_c[1])}/{len(peaks_radar_by_c[2])}",
    f"Radar xy (융합/클러스터): {len(radar_xy)} | LiDAR DBSCAN boxes: {len(lidar_boxes)}",
]
if FUSION_ERROR:
    lines.append("AI: " + FUSION_ERROR[:80])
ax.text(0, 1, "\n".join(lines), va="top", fontsize=10, family="monospace", transform=ax.transAxes)

# ----- row 1: BEV 패널 4개 -----
ax = fig.add_subplot(gs[1, 0])
plot_bev_background(ax, lidar_pts[:, :3], bev_cfg)
for x, y in radar_xy:
    ax.scatter(y, x, c="red", s=36, marker="x", zorder=5)
_bev_panel_tag(ax, "배경: LiDAR 점")
ax.set_title("③ BEV: LiDAR(회색) + Radar(X)")

cols = ["#ff7f0e", "#e377c2", "#17becf"]

ax = fig.add_subplot(gs[1, 1])
plot_bev_background(ax, lidar_pts[:, :3], bev_cfg)
for g in gt_footprints:
    poly = g["polygon_xy"]
    ax.add_patch(Polygon(poly[:, ::-1], closed=True, fill=False, edgecolor="lime", linewidth=1.8))
for c, pname in enumerate(class_names):
    first = True
    for x, y in peaks_lidar_by_c[c]:
        ax.scatter(y, x, c=cols[c], s=22, alpha=0.9, label=pname if first else "_nolegend_")
        first = False
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, labels, loc="upper right", fontsize=7)
_bev_panel_tag(ax, "학습 입력: LiDAR BEV")
ax.set_title("④ LiDAR BEV로 학습한 CNN + GT(초록)")

ax = fig.add_subplot(gs[1, 2])
plot_bev_background(ax, lidar_pts[:, :3], bev_cfg)
for g in gt_footprints:
    poly = g["polygon_xy"]
    ax.add_patch(Polygon(poly[:, ::-1], closed=True, fill=False, edgecolor="lime", linewidth=1.8))
if model_radar is not None:
    for c, pname in enumerate(class_names):
        first = True
        for x, y in peaks_radar_by_c[c]:
            ax.scatter(y, x, c=cols[c], s=22, alpha=0.9, label=pname if first else "_nolegend_")
            first = False
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles, labels, loc="upper right", fontsize=7)
    _bev_panel_tag(ax, "학습 입력: RADAR만")
else:
    ax.text(0.5, 0.5, "RADAR-CNN 없음\n(학습 샘플 부족)", ha="center", va="center", transform=ax.transAxes)
    _bev_panel_tag(ax, "RADAR 학습 스킵")
ax.set_title("⑤ 레이더만 BEV로 학습한 CNN + GT(초록)")

ax = fig.add_subplot(gs[1, 3])
plot_bev_background(ax, lidar_pts[:, :3], bev_cfg)
for (xm, ym), (xM, yM) in lidar_boxes:
    rect = mpatches.Rectangle(
        (ym, xm), yM - ym, xM - xm, fill=False, edgecolor="cyan", linewidth=1.5
    )
    ax.add_patch(rect)
_bev_panel_tag(ax, "규칙: DBSCAN")
ax.set_title("⑥ LiDAR DBSCAN 축정렬 박스")

fig.suptitle(
    f"[2×4] LiDAR-CNN vs RADAR-only-CNN vs YOLO — frame {FRAME_ID}",
    fontsize=13,
    fontweight="bold",
)
fig.text(
    0.5,
    0.005,
    "하단 행: ③융합배경 ④LiDAR학습CNN ⑤레이더만학습CNN ⑥DBSCAN",
    ha="center",
    fontsize=10,
    color="navy",
)
plt.tight_layout(rect=[0, 0.02, 1, 0.96])
plt.show()


---

### 참고 (용어·운영 정리)

- **`strong_validated` / `radar_only` 등 LiDAR 검증 태그**는 **운영 화면의 핵심 구분**으로 두기보다, **오프라인 분석**에 두고, 운영에서는 **`high_confidence` / `medium_confidence` / `low_confidence`**처럼 **레이더 특징 기반** 등급으로 치환하는 것을 권장합니다.  
- **학습 CNN** 두 종류는 동일 히트맵 타깃에 대해 **입력만** LiDAR BEV vs 레이더 BEV로 다릅니다 (3D 박스 직접 회귀 아님).  
- **GT 초록 다각형**은 KITTI `label_2` 바닥면을 velodyne으로 변환한 것입니다 (**B 구간 평가 기준**).  
- **YOLO**는 이미지 좌표계라 BEV와 직접 정합되지 않습니다 — **나란히 정성 비교**용입니다.  
- 전체 mAP·정밀한 레이더–카메라 정합은 별도 파이프라인이 필요합니다.
